In [1]:
from scipy import stats
import pandas as pd

df_eda = pd.read_csv("churn.csv").drop(columns=['RowNumber', 'CustomerId', 'Surname'])

In [2]:
categorical_cols = ['Geography', 'Gender', 'NumOfProducts', 'HasCrCard', 
                     'IsActiveMember', 'Card Type', 'Satisfaction Score']

chi_results = []

for col in categorical_cols:
    contingency = pd.crosstab(df_eda[col], df_eda['Exited'])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    chi_results.append({'Variable': col, 'Chi2': round(chi2, 2), 'p-value': p, 
                         'Significant (p<0.05)': p < 0.05})

chi_df = pd.DataFrame(chi_results)
print(chi_df)

             Variable     Chi2       p-value  Significant (p<0.05)
0           Geography   300.63  5.245736e-66                  True
1              Gender   112.40  2.925368e-26                  True
2       NumOfProducts  1501.50  0.000000e+00                  True
3           HasCrCard     0.45  5.026182e-01                 False
4      IsActiveMember   243.69  6.153167e-55                  True
5           Card Type     5.05  1.679411e-01                 False
6  Satisfaction Score     3.80  4.333650e-01                 False


Geography, Gender, NumOfProducts, and IsActiveMember are genuine churn drivers
HasCrCard, Card Type, and Satisfaction Score are not, the dataset's original assumption about HasCrCard (cardholders churn less) is now formally disproven, not just visually flat

In [3]:
numeric_cols = ['Age', 'Tenure', 'Balance', 'CreditScore', 'EstimatedSalary']

ttest_results = []

for col in numeric_cols:
    churned = df_eda[df_eda['Exited'] == 1][col]
    retained = df_eda[df_eda['Exited'] == 0][col]
    t_stat, p = stats.ttest_ind(churned, retained, equal_var=False)  # Welch's t-test
    ttest_results.append({'Variable': col, 't-stat': round(t_stat, 2), 'p-value': p,
                           'Significant (p<0.05)': p < 0.05})

ttest_df = pd.DataFrame(ttest_results)
print(ttest_df)

          Variable  t-stat        p-value  Significant (p<0.05)
0              Age   30.42  4.399452e-179                  True
1           Tenure   -1.35   1.771113e-01                 False
2          Balance   12.48   5.817634e-35                  True
3      CreditScore   -2.60   9.284913e-03                  True
4  EstimatedSalary    1.24   2.142820e-01                 False


CreditScore shows statistical significance (p=0.009) due to the large sample size, but the actual effect size is negligible (~7 points on an 850-point scale) and not practically meaningful for churn prediction.

In [4]:
contingency_complain = pd.crosstab(df_eda['Complain'], df_eda['Exited'])
chi2, p, dof, expected = stats.chi2_contingency(contingency_complain)
print(f"Complain vs Exited — Chi2: {chi2:.2f}, p-value: {p}")

Complain vs Exited — Chi2: 9907.91, p-value: 0.0
